# 01 — 在 Workbench 訓練 Iris 分類模型

在 OpenShift AI Workbench 中訓練 scikit-learn 模型，儲存至 PVC 供部署使用。

**Workbench 映像建議**：Jupyter | Data Science | CPU | Python 3.12（Version 3.4）  
**模型輸出路徑**：`/opt/app-root/src/models/model.pkl`

(可跳過)**前置**：請先執行 `00-setup-persistent-venv.ipynb`，並將 Kernel 切換為 **Python (dev-venv)**。

若暫時使用預設 Kernel，可在下一格執行：
```python
# 離線：Data Science 映像通常已有 sklearn／joblib → 直接跳過 pip
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

def _missing(*names):
    return [n for n in names if importlib.util.find_spec(n) is None]

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

need = _missing("sklearn", "joblib")
if not need:
    print("deps OK — skip pip")
else:
    wheel = os.environ.get("WHEELHOUSE") or os.environ.get("PIP_FIND_LINKS") or ""
    wheel_path = Path(wheel) if wheel else None
    if wheel_path and wheel_path.is_dir():
        _pip("--no-index", f"--find-links={wheel_path}", "scikit-learn", "joblib")
    elif os.environ.get("OFFLINE", "").lower() in ("1", "true", "yes"):
        raise SystemExit(
            f"OFFLINE=1 且缺少: {need}。請用已含 scikit-learn／joblib 的 Runtime Image，或設 WHEELHOUSE。"
        )
    else:
        print(f"missing {need} — trying online pip (will fail if air-gapped)")
        _pip("scikit-learn", "joblib")
```

In [ ]:
import os
from pathlib import Path

MODEL_DIR = Path("/opt/app-root/src/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "model.pkl"

print(f"Workbench 工作目錄: {os.getcwd()}")
print(f"PVC 根目錄: /opt/app-root/src/")
print(f"模型將儲存至: {MODEL_PATH}")

In [ ]:
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
joblib.dump(model, MODEL_PATH)
print(f"Model saved: {MODEL_PATH}")
print(f"File size: {MODEL_PATH.stat().st_size / 1024:.1f} KB")
assert MODEL_PATH.exists(), "模型檔案未成功建立"

In [ ]:
# 驗證本地推論
sample = [[5.1, 3.5, 1.4, 0.2]]
pred = model.predict(sample)[0]
print(f"Sample: {sample[0]}")
print(f"Prediction: {pred} ({iris.target_names[pred]})")

## 下一步：部署模型

1. 回到 **OpenShift AI Dashboard** → 專案 → **Deploy model**
2. 設定：
   - Model type: **Predictive**
   - Model framework: **Scikit-learn**
   - Source: **Existing cluster storage** → 選擇此 Workbench 的 PVC
   - Model path: `models/`
   - Deployment mode: **Standard**
3. 部署完成後，依 `02-test-inference.ipynb`：Dashboard 確認 Ready → Terminal `curl` 測試

詳細步驟見 [getting-started-ui-tutorial.ipynb](../docs-ipynb/getting-started-ui-tutorial.ipynb)（Deploy model）與 `02-test-inference.ipynb`。
